# 13 Tabular Export

This notebook audits TorchLens tabular exports. `to_pandas()` remains the in-memory table path, and canonical file export goes through `torchlens.export.csv`, `torchlens.export.json`, and `torchlens.export.parquet` for traces, records, and supported accessors.

The model has parameters, a buffer, multiple module records, and several layers so the DataFrame examples are meaningful but still small.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "audit":
    REPO_ROOT = REPO_ROOT.parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"repo root: {REPO_ROOT}")

In [ ]:
import json
from pathlib import Path

import pandas as pd
import torch
from torch import nn
import torchlens as tl

ARTIFACT_DIR = REPO_ROOT / "notebooks" / "audit" / "_artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

torch.manual_seed(59)


class TabularNet(nn.Module):
    """Small model with tabular metadata surfaces."""

    def __init__(self) -> None:
        """Initialize layers and a buffer."""

        super().__init__()
        self.encoder = nn.Linear(4, 5)
        self.decoder = nn.Linear(5, 2)
        self.register_buffer("offset", torch.tensor([0.25, -0.25]))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Run the encoder, activation, decoder, and buffer add.

        Parameters
        ----------
        x:
            Input tensor with four features.

        Returns
        -------
        torch.Tensor
            Two-feature output tensor.
        """

        hidden = torch.relu(self.encoder(x))
        return self.decoder(hidden) + self.offset


model = TabularNet().eval()
x = torch.randn(3, 4)
trace = tl.trace(model, x, layers_to_save="all")
print(f"layers: {len(trace.layers)}")
print(f"params: {len(trace.params)}")
print(f"buffers: {len(trace.buffers)}")
print(f"modules: {len(trace.modules)}")

`Trace.to_pandas()` gives one row per visible layer with a broad set of analysis columns.

In [ ]:
layer_df = trace.to_pandas()
print(layer_df[["layer_label", "layer_type", "shape", "activation_memory"]].to_string(index=False))

Once in pandas, ordinary sorting and filtering are enough for many audit questions.

In [ ]:
memory_rank = layer_df.sort_values("activation_memory", ascending=False)[
    ["layer_label", "layer_type", "activation_memory", "num_params"]
]
linear_only = layer_df[layer_df["layer_type"].str.contains("linear", case=False, na=False)][
    ["layer_label", "shape", "num_params"]
]
print("largest by activation memory:")
print(memory_rank.head(4).to_string(index=False))
print("\nlinear-like layers:")
print(linear_only.to_string(index=False))

Accessors export narrower tables for layers, modules, parameters, and buffers.

In [ ]:
surfaces = {
    "layers": trace.layers,
    "modules": trace.modules,
    "params": trace.params,
    "buffers": trace.buffers,
}
for name, surface in surfaces.items():
    frame = surface.to_pandas()
    print(
        f"{name:7s} rows={len(frame):2d} columns={len(frame.columns):2d} first_columns={list(frame.columns[:4])}"
    )

Individual records also have `to_pandas()`. This is useful when you want every field for one operation or one module call.

In [ ]:
first_linear = next(layer for layer in trace.layers if layer.func_name == "linear")
module_call = trace.module_calls["encoder:1"]
record_frames = {
    "layer_record": first_linear.to_pandas(),
    "module_call": module_call.to_pandas(),
}
for name, frame in record_frames.items():
    print(f"{name}: rows={len(frame)} columns={len(frame.columns)}")
    print(frame.iloc[:, :6].to_string(index=False))

The canonical file exporters are package functions: `torchlens.export.csv(log, path)`, `torchlens.export.json(log, path)`, and `torchlens.export.parquet(log, path)`. Parquet requires `pyarrow`, so this cell prints a note instead of failing when that optional dependency is absent.

In [ ]:
exports = {
    "trace": trace,
    "layers": trace.layers,
    "params": trace.params,
    "modules": trace.modules,
    "buffers": trace.buffers,
}
for name, surface in exports.items():
    base = ARTIFACT_DIR / f"13_{name}"
    csv_path = base.with_suffix(".csv")
    json_path = base.with_suffix(".json")
    parquet_path = base.with_suffix(".parquet")

    tl.export.csv(surface, csv_path)
    tl.export.json(surface, json_path)
    try:
        tl.export.parquet(surface, parquet_path)
        parquet_status = f"parquet rows={len(pd.read_parquet(parquet_path))}"
    except Exception as exc:
        parquet_status = f"> NOTE: parquet skipped: {type(exc).__name__}: {exc}"

    csv_rows = len(pd.read_csv(csv_path))
    json_rows = len(pd.read_json(json_path, orient="records"))
    print(f"{name:11s} csv_rows={csv_rows} json_rows={json_rows} {parquet_status}")

JSON files are regular records-oriented tables, so they are easy to inspect without pandas.

In [ ]:
json_path = ARTIFACT_DIR / "13_params.json"
records = json.loads(json_path.read_text())
print(f"json records: {len(records)}")
print({key: records[0][key] for key in ["address", "shape", "trainable"]})

The examples below stay on the canonical package exporter surface. Instance exporter methods are intentionally not used.

## Export Schemas and Round Trip
`torchlens.export.*` writes the same tabular schemas that `to_pandas()` exposes for supported surfaces. This cell checks representative trace and accessor columns after CSV/JSON round trips.

In [ ]:
export_surfaces = {
    "trace": trace,
    "layers": trace.layers,
    "modules": trace.modules,
    "params": trace.params,
    "buffers": trace.buffers,
}
frames = {}
for name, surface in export_surfaces.items():
    csv_path = ARTIFACT_DIR / f"13_{name}_schema.csv"
    json_path = ARTIFACT_DIR / f"13_{name}_schema.json"
    tl.export.csv(surface, csv_path)
    tl.export.json(surface, json_path)
    csv_frame = pd.read_csv(csv_path)
    json_frame = pd.read_json(json_path, orient="records")
    print(
        f"{name}: csv={len(csv_frame)} json={len(json_frame)} columns={list(csv_frame.columns[:6])}"
    )
    frames[name] = csv_frame

trace_columns = [
    column
    for column in ["layer_label", "func_name", "shape", "activation_memory"]
    if column in frames["trace"].columns
]
layer_columns = [
    column
    for column in ["layer_label", "func_name", "shape", "activation_memory"]
    if column in frames["layers"].columns
]
print(f"trace schema columns present: {trace_columns}")
print(f"layer schema columns present: {layer_columns}")
print(
    f"round trip consistent: {len(pd.read_csv(ARTIFACT_DIR / '13_trace_schema.csv')) == len(frames['trace'])}"
)
print(frames["trace"][["layer_label", "activation_memory"]].head(3).to_string(index=False))

## 🔧 Sandbox

Mini-experiment: change `sandbox_sort_key`, `sandbox_surface_name`, and `sandbox_descending`. Expected observation: row order and available columns depend on the selected surface, but export row counts match the DataFrame.

In [ ]:
sandbox_sort_key = "activation_memory"
sandbox_surface_name = "layers"
sandbox_descending = True
sandbox_surfaces = {
    "layers": trace.layers,
    "modules": trace.modules,
    "params": trace.params,
}
sandbox_surface = sandbox_surfaces[sandbox_surface_name]
sandbox_frame = sandbox_surface.to_pandas()
if sandbox_sort_key not in sandbox_frame.columns:
    sandbox_sort_key = sandbox_frame.columns[0]
sandbox_path = ARTIFACT_DIR / f"13_sandbox_{sandbox_surface_name}.csv"
tl.export.csv(sandbox_surface, sandbox_path)
print(f"surface: {sandbox_surface_name}")
print(f"rows exported: {len(pd.read_csv(sandbox_path))}, rows in DataFrame: {len(sandbox_frame)}")
print(
    sandbox_frame.sort_values(sandbox_sort_key, ascending=not sandbox_descending)
    .head(5)
    .to_string(index=False)
)